# 🌙 Dream-AI — Colab Pro

Uma IA de programação que **sonha** para ficar mais inteligente.

Este notebook:
1. Instala as dependências
2. Carrega um modelo base de ~1B (Qwen2.5-Coder-1.5B-Instruct) em 4-bit
3. Roda o **ciclo do sonho**: inventa → resolve → **verifica executando** → memoriza
4. **Consolida** (LoRA) as soluções verificadas
5. Compara o modelo antes/depois

**Honestidade:** só vira aprendizado o que passa na execução real. Sem alucinação.

> Use um runtime com **GPU** (Runtime → Change runtime type → T4/L4/A100).

## 1. Clonar o repositório e instalar

In [ ]:
!git clone https://github.com/felipe9272727/dream-ai.git
%cd dream-ai
!git checkout claude/1b-ai-from-scratch-nkhZK

!pip install -q torch numpy tokenizers
!pip install -q transformers datasets peft bitsandbytes accelerate

## 2. Sanidade: testar a maquinaria do sonho (sem GPU)

Antes de gastar GPU, confirmamos que o ciclo sonho→verifica→memória funciona.

In [ ]:
!python tests/test_dream.py
!python -m dream.loop --mode seed --dreams 20 --cycles 1

## 3. Carregar o modelo Coder de 1B

Qwen2.5-Coder-1.5B-Instruct em 4-bit cabe folgado no Colab Pro.

In [ ]:
from src.coder import CoderModel

coder = CoderModel(base_model="Qwen/Qwen2.5-Coder-1.5B-Instruct", load_in_4bit=True)
coder.load()

## 4. Testar o modelo ANTES de sonhar

In [ ]:
print(coder.solve("Escreva uma função Python `bubble_sort(lista)` que ordena uma lista."))

## 5. 🌙 Dormir e sonhar

O modelo inventa problemas, resolve, verifica executando, e memoriza os corretos.
Depois consolida tudo via LoRA. Acorda mais inteligente.

In [ ]:
import random
from dream.loop import run_cycle
from dream.consolidate import consolidate, load_memory

rng = random.Random(42)
for c in range(3):
    stats = run_cycle(coder, n_dreams=20, difficulty=2, rng=rng, use_model=True)
    print(f"Ciclo {c+1}: {stats}")

print(f"\nMemórias verificadas acumuladas: {len(load_memory())}")

# 😴 Consolidação (sono profundo)
adapter_path = consolidate(coder, epochs=2)

## 6. Auditar o que ele aprendeu (transparência)

A memória de sonhos é um arquivo legível — você vê exatamente o que virou aprendizado.

In [ ]:
from dream.consolidate import load_memory
for ex in load_memory()[:5]:
    print('—', ex['instruction'].splitlines()[0])
    print(ex['solution'])
    print()

## 7. Recarregar com o adapter aprendido e testar DEPOIS

In [ ]:
coder_v2 = CoderModel(
    base_model="Qwen/Qwen2.5-Coder-1.5B-Instruct",
    adapter_path="adapters/dream_lora",
    load_in_4bit=True,
)
coder_v2.load()
print(coder_v2.solve("Escreva uma função Python `bubble_sort(lista)` que ordena uma lista."))